# FastSpeech 2 — English & Telugu: training on Colab / Kaggle GPU

Runtime → *Change runtime type* → **T4 GPU** (or better). Everything (data, logs, checkpoints, results) is kept on Google Drive so a disconnected session can **resume** from `checkpoints/last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"   # <-- your fork / repo
WORK = "/content/drive/MyDrive/fastspeech2_project"


In [ ]:
import os, subprocess
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
%cd $WORK
!git pull --ff-only || true
!pip -q install -r requirements.txt
!python scripts/download_assets.py --vocoder --nltk
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Hugging Face token
Svarah is gated: open https://huggingface.co/datasets/ai4bharat/Svarah, accept the conditions, create a *read* token at https://huggingface.co/settings/tokens and paste it below (or store it as a Colab secret named `HF_TOKEN`).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = input("HF token: ").strip()

## Telugu

In [ ]:
!python Telugu/preprocess.py
!cat data/telugu/raw/selection_report.json | head -40

In [ ]:
# Re-run this cell after a disconnect: training resumes automatically.
!python Telugu/train.py

In [ ]:
!python Telugu/inference.py

## English

In [ ]:
!python English/preprocess.py
!cat data/english/raw/selection_report.json | head -40

In [ ]:
!python English/train.py

In [ ]:
!python English/inference.py

## Report
Listen to `Results/*/test_generated_audios/*.wav`, write your observations into `Report/observations.yaml`, then build the PDF.

In [ ]:
!python -m fastspeech2.cli report --config English/config.yaml
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('Results/*/test_generated_audios/*.wav')):
    print(f); display(Audio(f))

## Optional: TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs